In [9]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

In [10]:
VOCAB_SIZE = 10000

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 25000 | Test: 25000


In [11]:
MAX_LEN = 300   # 300 is enough — longer adds noise, not signal

X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='pre', truncating='pre')
X_test  = pad_sequences(X_test,  maxlen=MAX_LEN, padding='pre', truncating='pre')

print(f"X_train shape: {X_train.shape}")


X_train shape: (25000, 300)


In [12]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE,
              output_dim=128,        # bigger embedding = richer word representations
              input_length=MAX_LEN),

    Bidirectional(LSTM(64,           # Bidirectional reads sequence forwards AND backwards
                       return_sequences=False)),

    Dropout(0.5),                    # prevents overfitting

    Dense(32, activation='relu'),    # extra dense layer for better decision boundary
    Dropout(0.3),

    Dense(1, activation='sigmoid')   # final output: 0 = negative, 1 = positive
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [14]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=2,              # stops if val_accuracy doesn't improve for 2 epochs
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 167s 1s/step - accuracy: 0.7127 - loss: 0.5367 - val_accuracy: 0.8384 - val_loss: 0.3662
Epoch 2/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - accuracy: 0.8827 - loss: 0.2988 - val_accuracy: 0.8714 - val_loss: 0.3369
Epoch 3/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - accuracy: 0.9140 - loss: 0.2290 - val_accuracy: 0.8738 - val_loss: 0.3081
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - accuracy: 0.9344 - loss: 0.1800 - val_accuracy: 0.8686 - val_loss: 0.3959
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 202s 1s/step - accuracy: 0.9582 - loss: 0.1217 - val_accuracy: 0.8694 - val_loss: 0.4401


In [15]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Loss     : {loss:.4f}")
print(f"Test Accuracy : {accuracy:.4f}")


Test Loss     : 0.3211
Test Accuracy : 0.8687
